# Module 22 — Prompt Engineering as Bayesian Conditioning

**Part V · Generation & Alignment · 25–30 min**

---

There is a genre of LinkedIn post that lists "10 ChatGPT prompts that will 10x your productivity." There is a genre of $99 ebook that promises to teach you "the secret prompt formula." There is, on Twitter, an entire cottage industry of people screenshotting prompts in serif fonts and calling them "spells."

This module is here to ruin all of that.

A prompt is **not** a magic incantation. It's not a special password the model recognizes. There is no hidden lookup table mapping "You are an expert..." to a different model. The model has one set of weights and one job: predict the next token.

What a prompt actually does is much more boring and much more interesting:

> A prompt is a **conditioning event** on a probability distribution.

The model is computing $P(\text{next token} \mid \text{everything before})$. The prompt *is* the everything before. Change the prompt, change the conditioning, change the distribution. That's it. That's the whole trick. The "art" of prompt engineering is just learning what kinds of context shift the distribution toward outputs you want.

Once you internalize this, three things happen:

1. You stop believing in magic phrases.
2. You start being able to *predict* which prompts will work, instead of cargo-culting other people's.
3. You realize that prompting and alignment training (Modules 20–21) are doing the same thing from opposite ends. Prompting reshapes the distribution at inference time. Alignment reshapes it at training time. One is cheap and reversible; the other is expensive and permanent.

We're going to make this concrete by feeding GPT-2 the same question over and over and watching the probability mass slosh around as we change the conditioning. By the end you should never look at a "system prompt" the same way again.

## 0 · Setup

Plain PyTorch, `transformers`, GPT-2. Runs on CPU in a couple of minutes. We use GPT-2 small precisely because it is dumb enough that we can *see* the conditioning effects without them being washed out by years of RLHF.

In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tok = GPT2Tokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
model.eval()
print("loaded gpt2 (124M)")

## 1 · The Bayesian re-framing

Here is the equation everyone half-remembers from a stats class:

$$
P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B)}
$$

Read aloud: "the probability of A given B is proportional to how likely B is when A is true, times how likely A was to begin with."

Now substitute LLM things. Let $a$ be a candidate answer, $p$ be the prompt:

$$
\underbrace{P(a \mid p)}_{\text{posterior}} \;\propto\; \underbrace{P(p \mid a)}_{\text{likelihood}}\;\cdot\;\underbrace{P(a)}_{\text{prior}}
$$

Three pieces. Each one means something specific:

- **Prior $P(a)$** — how plausible is this answer in the abstract, before you said anything? This is "what does the model believe about the world by default, given how often this answer appeared in pretraining?" If you ask "what's the capital of Australia?" with no context, $P(\text{Sydney})$ is unfortunately high because Sydney is mentioned **a lot** in training data and Canberra basically isn't.

- **Likelihood $P(p \mid a)$** — given that the answer really is $a$, how likely is it that someone would have written *this exact prompt* in front of it? "You are a careful geography expert. Q: What is the capital of Australia? A:" is the kind of context that almost never precedes the answer "Sydney" in well-edited training data, but very plausibly precedes "Canberra." That's the likelihood working for you.

- **Posterior $P(a \mid p)$** — what the model actually outputs after seeing your prompt. This is the only thing you can directly observe in the logits.

The model does not literally compute Bayes rule. It learned the joint $P(\text{tokens})$ during pretraining and is sampling from a conditional. But because Bayes rule is *true*, the same factorization applies. **Prompting is the act of providing context $p$ such that the implied likelihood $P(p \mid a^*)$ for your desired answer $a^*$ is much higher than for the wrong answers.** That's all "prompt engineering" really is. The rest is taste.

## 2 · Plumbing: getting next-token probabilities

We need a tiny utility that takes a prompt, runs one forward pass, and tells us the probability the model assigns to specific candidate continuations. Nothing fancy — just softmax over the last logit row, then look up the tokens we care about.

A subtlety: " Sydney" and "Sydney" are *different tokens* in GPT-2 (the leading space matters because BPE). We always include the leading space, because in practice the model is going to predict the version-with-space when continuing a sentence.

In [ ]:
@torch.no_grad()
def next_token_probs(prompt, candidates):
    """Return P(candidate | prompt) for each candidate string.

    candidates: list of strings, each must encode to exactly one token.
    """
    ids = tok.encode(prompt, return_tensors="pt").to(device)
    logits = model(ids).logits[0, -1]          # (vocab,)
    probs = F.softmax(logits, dim=-1)
    out = {}
    for c in candidates:
        c_ids = tok.encode(c)
        assert len(c_ids) == 1, f"{c!r} -> {c_ids}, must be 1 token"
        out[c] = probs[c_ids[0]].item()
    return out

@torch.no_grad()
def topk_next(prompt, k=8):
    ids = tok.encode(prompt, return_tensors="pt").to(device)
    logits = model(ids).logits[0, -1]
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    return [(tok.decode([i.item()]), p.item()) for i, p in zip(top.indices, top.values)]

# Sanity check
for tk, p in topk_next("The capital of France is", k=5):
    print(f"  {tk!r:>15s}   {p:.4f}")

Good. Paris dominates, as it should. GPT-2 is dumb but not *that* dumb.

Now let's break it.

## 3 · The Australia disaster

Here's the canonical example. GPT-2 — and, embarrassingly, several much larger models in their early days — gets the capital of Australia wrong. Sydney is the obvious answer because Sydney shows up in training data constantly: Olympic games, opera house, beaches, harbor, fires. Canberra shows up almost exclusively in dry geography texts.

So the **prior** $P(\text{Sydney})$ crushes $P(\text{Canberra})$. We have to fix this with conditioning.

In [ ]:
prompt0 = "Q: What is the capital of Australia?\nA:"
candidates = [" Sydney", " Canberra", " Melbourne", " Brisbane", " Perth"]

probs0 = next_token_probs(prompt0, candidates)
for c, p in probs0.items():
    print(f"  {c!r:>12s}   P = {p:.4f}   logp = {math.log(p):+.2f}")
print()
print("top-8 the model actually wants to say next:")
for tk, p in topk_next(prompt0, k=8):
    print(f"  {tk!r:>15s}   {p:.4f}")

Look at that. With a bare zero-shot question, GPT-2 doesn't even put Canberra in the top of the candidate set — Sydney is winning, possibly badly. (Your exact numbers will depend on tokenizer/model versions, but the qualitative story is almost always the same.)

This is **not** because the model is "wrong about geography" in some deep sense. The model learned the joint distribution of tokens on the internet. On the internet, in the contexts where someone writes "Q: What is the capital of Australia? A:", the next word is *often* Sydney, because the question is famously a trick question that people post specifically so that they can dunk on someone for getting it wrong. The model is doing exactly what it was trained to do. It just isn't a geography expert. **Yet.**

## 4 · Conditioning move #1: a system prompt

Let's tell the model who it is. This is the cheapest possible intervention — we shove a sentence in front and recompute the distribution.

In [ ]:
prompt1 = (
    "You are a careful, accurate geography expert. "
    "You answer with the official capital city, not the largest city.\n"
    "Q: What is the capital of Australia?\nA:"
)

probs1 = next_token_probs(prompt1, candidates)
for c in candidates:
    a, b = probs0[c], probs1[c]
    arrow = "↑" if b > a else "↓"
    print(f"  {c!r:>12s}   {a:.4f}  ->  {b:.4f}   {arrow}")

You should see Canberra's probability move *up* and Sydney's move *down*, even though we never said the word "Canberra" or "Sydney" in the prompt. That's the likelihood term doing work: the phrase "official capital city, not the largest city" is the kind of thing that, in the training distribution, is *much* more likely to precede "Canberra" than "Sydney." We tilted the implicit $P(p \mid a)$ in our favor.

Whether GPT-2 actually flips its top prediction depends on which version you load — sometimes it does, sometimes Sydney is still narrowly winning. That's fine. The point isn't the absolute value, it's the *direction of the shift*. Conditioning is a force, not a hammer.

## 5 · Conditioning move #2: one-shot example

Now let's give it an example of the format we want. One-shot prompting. We don't even use Australia in the example; we use a *different* country, so we're conditioning on a pattern, not leaking the answer.

In [ ]:
prompt2 = (
    "You are a careful, accurate geography expert.\n"
    "Q: What is the capital of Brazil?\nA: Brasilia\n"
    "Q: What is the capital of Australia?\nA:"
)

probs2 = next_token_probs(prompt2, candidates)
for c in candidates:
    a, b, cc = probs0[c], probs1[c], probs2[c]
    print(f"  {c!r:>12s}   zero={a:.4f}   sys={b:.4f}   one-shot={cc:.4f}")

Brazil/Brasilia is a beautifully chosen example because Brazil has the *exact same property*: the capital is not the most famous city. (You could have used São Paulo as the trap.) By showing the model "ah, here's a country where I picked the boring official capital, not the famous beach city," you condition it into a regime where picking the boring official capital is the locally-likely behavior. Canberra should jump again.

Notice what we did *not* do: we did not retrain the model. We did not give it new information. We didn't even mention Australia in the example. We just rearranged tokens in the context and the entire distribution slid. That is the whole power of in-context learning, and it is fundamentally a Bayesian update.

## 6 · Visualizing the slide

Three prompts, five candidates, side by side. This plot is the whole module in one figure.

In [ ]:
prompts = {
    "zero-shot":      probs0,
    "+ system":       probs1,
    "+ one-shot":     probs2,
}

labels = [c.strip() for c in candidates]
x = np.arange(len(labels))
width = 0.27

fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (name, pr) in enumerate(prompts.items()):
    vals = [pr[c] for c in candidates]
    ax.bar(x + (i-1)*width, vals, width, label=name)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("P(token | prompt)")
ax.set_title("How conditioning reshapes P(answer) — same question, three contexts")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

Stare at this until it feels obvious.

The probability mass *moved*. The model did not learn anything; we only gave it different stuff to condition on. Each bar group is the same softmax over the same vocabulary computed by the same weights — only the prefix differs.

If you came into this module thinking "system prompts are vibes," good news: they're not vibes, they are measurable interventions on a probability distribution, and you just measured one.

## 7 · Logprob view: the same story, but cleaner

Probabilities crowd near zero and are hard to compare across orders of magnitude. Log-probabilities are usually the better way to read what a language model "believes." A jump from $10^{-4}$ to $10^{-2}$ looks like nothing on a bar chart and like 4.6 nats on a log chart. That 4.6 nats is the actual scale of "how much did this prompt move the model's mind."

We'll measure the **logprob shift** for Canberra specifically, across a series of prompts of increasing helpfulness.

In [ ]:
prompt_progression = [
    ("zero-shot",
     "Q: What is the capital of Australia?\nA:"),
    ("+ \"capital city\"",
     "Q: What is the capital city of Australia?\nA:"),
    ("+ system role",
     "You are a geography expert.\nQ: What is the capital of Australia?\nA:"),
    ("+ \"not largest\"",
     "You are a geography expert. Answer with the official capital, not the largest city.\n"
     "Q: What is the capital of Australia?\nA:"),
    ("+ one-shot (Brazil)",
     "You are a geography expert. Answer with the official capital, not the largest city.\n"
     "Q: What is the capital of Brazil?\nA: Brasilia\n"
     "Q: What is the capital of Australia?\nA:"),
    ("+ two-shot (Brazil, USA)",
     "You are a geography expert. Answer with the official capital, not the largest city.\n"
     "Q: What is the capital of Brazil?\nA: Brasilia\n"
     "Q: What is the capital of the United States?\nA: Washington\n"
     "Q: What is the capital of Australia?\nA:"),
]

names, logp_canberra, logp_sydney = [], [], []
for name, p in prompt_progression:
    pr = next_token_probs(p, [" Canberra", " Sydney"])
    names.append(name)
    logp_canberra.append(math.log(pr[" Canberra"]))
    logp_sydney.append(math.log(pr[" Sydney"]))
    print(f"  {name:<25s}  logP(Canberra)={logp_canberra[-1]:+6.2f}   logP(Sydney)={logp_sydney[-1]:+6.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
xs = np.arange(len(names))
ax.plot(xs, logp_canberra, "o-", lw=2, label="Canberra (correct)", color="C2")
ax.plot(xs, logp_sydney,   "o-", lw=2, label="Sydney (wrong)",     color="C3")
ax.set_xticks(xs); ax.set_xticklabels(names, rotation=20, ha="right")
ax.set_ylabel("log P(token | prompt)")
ax.set_title("Each conditioning event shifts the distribution")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Two lines, walking apart. Canberra climbs, Sydney falls (or at least stops winning). Each step on the x-axis is one extra conditioning event. None of them taught the model anything; they all only changed what it was conditioning on.

This is also a really nice illustration of *diminishing returns*. The first few conditioning moves do most of the work. By the time you're adding a third or fourth few-shot example, you're squeezing nats out of a stone. The model has more or less committed.

## 8 · Chain-of-thought as marginalization

Here's the equation that makes "let's think step by step" feel inevitable instead of magical.

The model is trying to compute $P(a \mid Q)$ — the probability of answer $a$ given question $Q$. By the law of total probability, you can introduce a hidden variable $r$ ("a chain of reasoning") and marginalize over it:

$$
P(a \mid Q) \;=\; \sum_{r} P(a \mid r, Q)\, P(r \mid Q)
$$

In English: the probability of the right answer is the sum, over all possible reasoning chains, of (how likely that chain is given the question) times (how likely the right answer is once you've worked through that chain).

A model that is forced to commit to an answer in one token has no chance to evaluate this sum. It has to collapse to its single best guess of $a$ given $Q$ directly — which, for any non-trivial question, is just whatever $P(a \mid Q)$ peaks at. Often that's wrong.

A model that is allowed (or asked) to *write down* a chain of reasoning $r$ before committing is approximately doing this:

1. Sample one or a few high-probability reasoning chains $r$ from $P(r \mid Q)$.
2. Conditional on each chain, the answer distribution $P(a \mid r, Q)$ is much sharper and usually much more correct, because the chain has done useful intermediate work.
3. The final token is sampled from a distribution that is implicitly weighted by the likelihood of the chain.

So "let's think step by step" is **not a spell**. It's a context that puts the model in a regime where it spends its decoder budget on generating $r$ before committing to $a$. More tokens before the answer = more compute spent = better marginalization. This is also exactly the bridge to Module 24 (test-time compute), where we make this explicit and quantitative.

## 9 · Few-shot vs zero-shot vs CoT: a controlled bake-off

GPT-2 is going to be bad at arithmetic. That's *good*: a near-ceiling task can't show conditioning effects. We want a task where each intervention can move the needle. We'll use simple two-digit addition, which GPT-2 famously fails at but doesn't *catastrophically* fail at.

We'll measure two things:
1. **accuracy** (did the first numeric token after "A:" match the right answer?),
2. **average response length** (proxy for "how much compute did the model spend before committing?").

In [ ]:
import re

rng = np.random.default_rng(0)
N = 30
problems = [(int(rng.integers(10, 50)), int(rng.integers(10, 50))) for _ in range(N)]
truths = [a + b for a, b in problems]

@torch.no_grad()
def generate(prompt, max_new=40):
    ids = tok.encode(prompt, return_tensors="pt").to(device)
    out = model.generate(
        ids, max_new_tokens=max_new, do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def first_int(s):
    m = re.search(r"-?\d+", s)
    return int(m.group()) if m else None

def eval_prompt_template(template_fn, max_new=40):
    correct, lengths = 0, []
    for (a, b), t in zip(problems, truths):
        p = template_fn(a, b)
        gen = generate(p, max_new=max_new)
        # CoT: take whatever number the model commits to *last* before stopping or after "answer is"
        ans = first_int(gen.split("\n")[0]) if "step by step" not in p.lower() else None
        if ans is None:
            ans = first_int(gen)
        if ans == t:
            correct += 1
        lengths.append(len(tok.encode(gen)))
    return correct / len(problems), float(np.mean(lengths))

zero = lambda a, b: f"Q: What is {a} + {b}?\nA:"
fewshot = lambda a, b: (
    "Q: What is 13 + 24?\nA: 37\n"
    "Q: What is 47 + 31?\nA: 78\n"
    "Q: What is 19 + 22?\nA: 41\n"
    f"Q: What is {a} + {b}?\nA:"
)
cot = lambda a, b: (
    f"Q: What is {a} + {b}?\n"
    "A: Let's think step by step. "
)

results = {}
for name, fn, max_new in [
    ("zero-shot", zero, 8),
    ("3-shot",    fewshot, 8),
    ("CoT",       cot, 60),
]:
    acc, length = eval_prompt_template(fn, max_new=max_new)
    results[name] = (acc, length)
    print(f"  {name:<10s}  acc={acc:.2f}   avg_len={length:.1f} tokens")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
names = list(results.keys())
acc = [results[n][0] for n in names]
length = [results[n][1] for n in names]

axes[0].bar(names, acc, color=["C0", "C1", "C2"])
axes[0].set_ylabel("accuracy"); axes[0].set_ylim(0, 1)
axes[0].set_title("accuracy: zero vs few-shot vs CoT")
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(names, length, color=["C0", "C1", "C2"])
axes[1].set_ylabel("avg generated tokens")
axes[1].set_title("compute spent (= tokens before committing)")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.show()

A few honest caveats about what you just saw:

1. GPT-2 is genuinely bad at arithmetic. Don't expect dramatic numbers; expect *directional* numbers. CoT will usually beat zero-shot, few-shot will usually beat zero-shot, and CoT will spend dramatically more tokens. With a real frontier model the gaps blow open and CoT often doubles accuracy on this kind of task.

2. The point is the **shape** of the trade-off, not the absolute heights of the bars. CoT trades tokens (i.e. compute, i.e. money, i.e. latency) for accuracy. That trade is the entire premise of test-time compute scaling, which is Module 24.

3. We're using greedy decoding so the experiment is reproducible. With sampling + self-consistency (Wang et al. 2022) you can push CoT accuracy substantially higher by sampling many reasoning chains and taking a majority vote — which is *literally* a Monte Carlo estimate of the marginalization sum from section 8. If the math feels too theoretical, that's the bridge: self-consistency is just doing the sum on purpose.

## 10 · Break-things-on-purpose: misleading few-shot

Conditioning is a force that doesn't care which direction you push. If you push wrong, the model goes wrong, *enthusiastically*.

Let's give the model a few-shot prompt where the in-context examples are subtly broken: every answer is the sum minus one. Watch what happens.

In [ ]:
bad_fewshot = lambda a, b: (
    "Q: What is 13 + 24?\nA: 36\n"   # off by 1
    "Q: What is 47 + 31?\nA: 77\n"   # off by 1
    "Q: What is 19 + 22?\nA: 40\n"   # off by 1
    f"Q: What is {a} + {b}?\nA:"
)

acc_bad, len_bad = eval_prompt_template(bad_fewshot, max_new=8)
print(f"  bad 3-shot  acc(vs true sum)         = {acc_bad:.2f}")

# How often did it match the *poisoned* pattern (sum - 1)?
correct_bad_pattern = 0
for (a, b), t in zip(problems, truths):
    gen = generate(bad_fewshot(a, b), max_new=8)
    ans = first_int(gen)
    if ans == t - 1:
        correct_bad_pattern += 1
print(f"  bad 3-shot  match-the-poisoned-rule  = {correct_bad_pattern/N:.2f}")

Two numbers worth staring at:

- Accuracy against the *true* sum drops, sometimes to zero.
- Match-rate against the *poisoned* sum-minus-one rule rises, often substantially.

The model is not "wrong." The model is doing exactly the same thing it was doing before: pattern-matching the in-context examples and continuing them. We changed the examples, so the pattern changed, so the answers changed. The conditioning event is symmetric with respect to truth. **There is nothing in the prompting machinery that makes the model prefer correct examples over consistent ones.**

This is also basically how prompt injection attacks work in production. An attacker drops a few "examples" into a context window — through a tool result, a retrieved document, a user message — and the model dutifully shifts its distribution to match. Conditioning is conditioning. The model can't tell the trustworthy bytes from the untrustworthy ones.

## 11 · Bonus: peeking at the full top-k under each prompt

Sometimes it's more illuminating to just *look* at the top of the distribution under different conditioning. We'll dump the top-5 next-token candidates for our progression and let you eyeball the personality changes.

In [ ]:
for name, p in prompt_progression:
    print(f"\n--- {name} ---")
    print(f"prompt: {p[-80:]!r}")
    for tk, pr in topk_next(p, k=5):
        print(f"   {tk!r:>15s}  {pr:.4f}")

Notice how the **shape** of the distribution changes too — not just which token is on top, but how concentrated the mass is. A vague prompt gives a flat top-k (high entropy: the model is unsure). A specific, well-conditioned prompt gives a peaked top-k (low entropy: the model has been backed into a corner). Entropy of the next-token distribution is a surprisingly good proxy for "how committed is the model right now."

If you ever want to write a quick prompt evaluator, measuring the entropy of the output distribution across a fixed test set is one of the most underrated tools you can build. You don't even need ground truth labels. Lower entropy at the answer position usually correlates with the model knowing what it's doing.

## 12 · The bridge: prompting vs alignment

Here's the punchline that ties this module to the rest of Part V.

Everything we did in this notebook was **inference-time conditioning**. We never touched the model weights. We just changed the bytes in front of the model and watched the distribution move.

Alignment training (Modules 20–21: SFT, DPO, GRPO) is doing **the same shaping operation, but at training time, by changing the weights**. Instead of having to type "You are a careful, accurate geography expert" every single time, you train the model so that the weights themselves implement that conditioning by default. The base distribution $P(a)$ — the "prior" in our Bayes equation — gets reshaped. After alignment, the model's prior is closer to "be helpful and accurate," so you don't have to spend prompt tokens dragging it there.

Two ways to look at the same trade-off:

| | Prompting | Alignment training |
|---|---|---|
| **What changes** | the context | the weights |
| **When** | inference | training |
| **Cost** | $0, instant | $$$, days |
| **Reversible** | yes (next request) | no (next checkpoint) |
| **Per-user customization** | trivial | hard |
| **Survives a bad input** | no | partly |

The reason modern frontier models need *less* prompt engineering than GPT-2 is not that they "understand" prompts better. It's that alignment training has already absorbed all the conditioning that you used to have to do by hand. "You are a helpful assistant" is baked into the weights now. You no longer have to type it — but it's still happening, and if you ever load a base model (no SFT, no RLHF) you'll feel the difference instantly. A base model is a model where you have to do *all* the conditioning yourself.

So the lesson is not "prompting is dead." The lesson is: prompting and alignment are the same operation on the same distribution, applied at different points in the pipeline. The more you align, the less you have to prompt. The less you align, the more you have to prompt. The Bayes equation doesn't care which side of the training/inference line you're on.

## 13 · Checkpoint quiz

Answer these *before* peeking. They're conceptual, not computational.

1. **Why does adding "You are a geography expert" to a prompt sometimes change the answer, even though the new sentence contains no factual content about Australia?** What term in the Bayes equation moves, and why?

2. **Chain-of-thought as marginalization** — write down, in your own words, what $\sum_r P(a \mid r, Q) P(r \mid Q)$ is approximating, and explain why a single greedy decode is a bad approximation to that sum.

3. **The poisoned few-shot experiment** showed the model happily learning a wrong pattern from in-context examples. What does that imply about trusting any data that ends up in your context window — including retrieved documents, tool outputs, and user messages?

4. **Why does an aligned model (GPT-4, Claude) need less prompt engineering than GPT-2?** Frame your answer in terms of where the conditioning has been "moved to."

5. **(Stretch.)** Self-consistency (Wang et al. 2022) samples many CoT chains and majority-votes the final answer. Explain why this is literally a Monte Carlo estimator of the marginalization integral from section 8. What does increasing the number of samples buy you, in terms of the bias/variance of that estimate?

## 14 · What you should walk away with

- A prompt is not a magic phrase. A prompt is a context, and a context is a conditioning event on $P(\text{output} \mid \text{context})$.
- Bayes rule isn't a metaphor here, it's the actual factorization. Prior, likelihood, posterior — all three have concrete meanings in LLM-land, and you can move them on purpose.
- "Let's think step by step" works because it shoves the model into a regime where it spends more tokens generating intermediate reasoning before committing to an answer. That's marginalization over reasoning chains, approximately, on the cheap.
- Few-shot examples are a Bayesian update: each example sharpens the posterior. They cut both ways. Poisoned examples poison the posterior.
- Alignment training (Part V's other modules) is the same shaping operation, applied at training time to the weights instead of at inference time to the context. Modern aligned models need less prompt engineering because someone else already did the conditioning for you and baked it in.

Next up in Part V: how that baking-in actually happens (DPO, GRPO), how the alignment stack is layered, and how all of it gets attacked by red-teamers who treat your prompt as just another input to manipulate.

Now go delete the screenshots of "10 magic prompts" from your downloads folder.